# Interactive Demo


## Setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/NLP PROJECT/Neural_Search_Engine-main'
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
    !pip install -q rank-bm25
else:
    PROJECT_ROOT = os.path.abspath('..')
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

import json
import torch
import string
import textwrap
import pandas as pd
from rank_bm25 import BM25Okapi

from src.tokenizer import BPETokenizer
from src.model import BiEncoder
from src.vector_store import VectorStore

print('Setup complete.')

---
## 1. Load Corpus, BM25 Index and Fine-Tuned BiEncoder

In [ ]:
# Corpus
with open('data/processed/jurafsky_chunks_v2.json', encoding='utf-8') as f:
    chunks = json.load(f)
print(f'Loaded {len(chunks)} chunks from Jurafsky & Martin')

# BM25 baseline
def _tokenize(text: str):
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return text.split()

corpus    = [c['content'] for c in chunks]
chunk_ids = [c['id']      for c in chunks]
bm25      = BM25Okapi([_tokenize(d) for d in corpus])
print('BM25 index ready.')

# Trained from-scratch neural encoder + its tokenizer (saved by notebook 02)
tokenizer = BPETokenizer.load('checkpoints/tokenizer.json')
model = BiEncoder.load('checkpoints/best.pt')

store = VectorStore(model, tokenizer, batch_size=64)
store.build(chunks)
print('Neural encoder ready.')

---
## 2. Define a Pretty-Print Search Function

In [ ]:
def bm25_search(query: str, top_k: int = 3):
    """Return list of (chunk_id, score, content) tuples."""
    scores = bm25.get_scores(_tokenize(query))
    top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [(chunk_ids[i], round(scores[i], 3), corpus[i]) for i in top]


def show_results(query: str, top_k: int = 3, show_chars: int = 350):
    """Pretty-print BM25 and neural-encoder results side-by-side."""
    print('=' * 100)
    print(f'QUERY: {query}')
    print('=' * 100)

    bm25_results = bm25_search(query, top_k)
    bi_results   = store.search(query, top_k)

    # BM25 column
    print('\n-- BM25 (keyword baseline) ----------------------------------------------')
    for rank, (cid, score, content) in enumerate(bm25_results, 1):
        snippet = textwrap.shorten(content, width=show_chars, placeholder='...')
        print(f'  [{rank}] {cid}  score={score}')
        print(textwrap.indent(textwrap.fill(snippet, width=92), '      '))
        print()

    # Neural column
    print('-- Neural encoder (from-scratch, contrastively trained) -----------------')
    for r in bi_results:
        snippet = textwrap.shorten(r['content'], width=show_chars, placeholder='...')
        print(f"  [{r['rank']}] {r['id']}  score={r['score']}")
        print(textwrap.indent(textwrap.fill(snippet, width=92), '      '))
        print()

---
## 3. Try a Keyword-Friendly Query

BM25 should be competitive here because the query and the answer share many words.

In [ ]:
show_results('What is a bigram language model?', top_k=3)

---
## 4. Try a Semantic / Paraphrase Query

Now phrase a question using words that **don't** appear verbatim in the textbook.
The BiEncoder should still find the right chunk; BM25 will likely fail.

In [ ]:
show_results('How do machines understand the meaning of a sentence?', top_k=3)

In [ ]:
show_results('Why do we split text into subword pieces?', top_k=3)

In [ ]:
show_results('What measures how surprised a language model is by new text?', top_k=3)
# Hint: "perplexity" — the query doesn't say that word, but the answer chunk should.

---
## 5. Interactive Loop (run the cell, then type queries)

In [ ]:
print('Type a query and press ENTER. Type "quit" to exit.\n')
while True:
    try:
        q = input('Query > ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\nBye.')
        break
    if q.lower() in {'quit', 'exit', ''}:
        print('Bye.')
        break
    show_results(q, top_k=3)
    print()

---
## 6. Save the Vector Store for Faster Reuse

The chunk embeddings only need to be computed once. Save them so future runs
skip the encoding step (loading is < 1 s vs ~15 s rebuild on CPU).

In [ ]:
store.save('data/processed/vector_store')

# How to load it next time:
# store2 = VectorStore(model, tokenizer)
# store2.load('data/processed/vector_store')

---
## Notes for the Report

When writing up your findings, look for queries where:
- **BM25 wins** → query uses exact textbook terminology (good for showing BM25's strength on technical text)
- **BiEncoder wins** → paraphrased question or different vocabulary (good for showing the neural model's semantic understanding)
- **Both fail** → topic isn't well covered in the corpus, or the chunking split context badly

These three categories form the basis of your qualitative analysis section.